# Lesson 1A: Markov Decision Processes - Theory

## Introduction

Now that we understand what RL is and how environments work, we need **mathematical foundations** to reason rigorously about learning and decision-making.

Enter the **Markov Decision Process (MDP)**—the formal mathematical model underlying all RL algorithms. An MDP is a tuple (S, A, P, R, γ) that captures:
- What states the agent can be in
- What actions the agent can take
- How the world transitions between states
- What rewards the agent receives
- How much we value future rewards vs. immediate ones

If you can model a problem as an MDP, you can apply RL algorithms to solve it.

In this lesson, we'll:
1. Understand the **Markov property** and Markov chains
2. Define the **MDP tuple** formally
3. Explain **state-transition probabilities** and reward functions
4. Justify the **discount factor** mathematically
5. Derive the **Bellman equations** (the backbone of RL)
6. Implement an **MDP solver from scratch**
7. Analyze classic examples (Student MDP, Recycling Robot)

Then in Lesson 1B, we'll implement policy evaluation and iteration algorithms to solve MDPs.

## Table of Contents

1. [Required Libraries](#required-libraries)
2. [The Markov Property](#markov-property)
3. [Markov Chains](#markov-chains)
4. [Markov Decision Processes](#mdp-definition)
5. [State-Transition Probabilities](#state-transitions)
6. [Reward Functions](#rewards)
7. [The Discount Factor](#discount-factor)
8. [Value Functions](#value-functions)
9. [Bellman Equations](#bellman-equations)
10. [From-Scratch MDP Solver](#mdp-solver)
11. [Classic MDP Examples](#examples)
12. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

| Library | Purpose |
|---------|----------|
| NumPy | Matrix operations and probability distributions |
| Matplotlib | Visualization of MDPs and value functions |
| Networkx | Graph visualization of MDP structure |
| Pandas | Tabular display of transitions and values |

In [ ]:
import subprocess
import sys

# Install networkx if needed
try:
    import networkx as nx
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "networkx", "-q"])

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import networkx as nx
from typing import Dict, List, Tuple, Optional
from matplotlib.patches import FancyArrowPatch, Circle
from matplotlib.patches import FancyBboxPatch

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ All libraries loaded!")

<a name="markov-property"></a>
## The Markov Property

### Definition

A stochastic process has the **Markov property** if:

$$P(s_{t+1} | s_t, s_{t-1}, s_{t-2}, ..., s_0) = P(s_{t+1} | s_t)$$

**In words**: The probability of the next state depends **only on the current state**, not on the entire history. The future is independent of the past given the present.

### Why It Matters

Without the Markov property, RL would be intractable:
- We'd need to track the entire history of states
- We couldn't learn compact representations
- Planning would require exponentially more computation

With the Markov property:
- The current state is sufficient for decision-making
- Algorithms are tractable and scalable
- We can use dynamic programming

### Examples

**Markov**: Chess position determines best move (doesn't matter how we got here)

**Non-Markov**: Text completion ("The cat sat on the __" needs context beyond current word)

**Trick**: Non-Markov problems can be made Markov by augmenting the state:
- If you need the previous N words, make state = (last N words, current word)
- If you need acceleration history, include velocity in state

In [ ]:
# Demonstrate Markov property
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Markov (memoryless) process
ax = axes[0]
ax.text(0.5, 0.95, 'Markov: Memoryless', ha='center', fontsize=13, fontweight='bold',
        transform=ax.transAxes)

# Draw state sequence
states = ['s₀', 's₁', 's₂', 's₃']
positions = np.linspace(0.1, 0.9, len(states))

for i, (pos, state) in enumerate(zip(positions, states)):
    circle = Circle((pos, 0.7), 0.05, transform=ax.transAxes, 
                   color='lightblue' if i < len(states)-1 else 'lightcoral', 
                   ec='black', linewidth=2)
    ax.add_patch(circle)
    ax.text(pos, 0.7, state, ha='center', va='center', transform=ax.transAxes,
           fontsize=11, fontweight='bold')
    
    if i < len(states) - 1:
        ax.annotate('', xy=(positions[i+1]-0.05, 0.7), xytext=(pos+0.05, 0.7),
                   arrowprops=dict(arrowstyle='->', lw=2), xycoords=ax.transAxes)

# Add formula
ax.text(0.5, 0.5, r'$P(s_{t+1} | s_t) = P(s_{t+1} | s_t, s_{t-1}, s_{t-2}, ...)$',
       ha='center', fontsize=12, transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax.text(0.5, 0.25, 'Decision depends only on current state\nNo need to remember history',
       ha='center', fontsize=11, transform=ax.transAxes, style='italic')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# Non-Markov (with history)
ax = axes[1]
ax.text(0.5, 0.95, 'Non-Markov: Memory-dependent', ha='center', fontsize=13, fontweight='bold',
        transform=ax.transAxes)

# Show why history matters
ax.text(0.5, 0.8, 'Example: Text generation',
       ha='center', fontsize=11, fontweight='bold', transform=ax.transAxes)

example1 = '"The bank robber escaped with" → next word?'
example2 = '"I went to the bank to" → next word?'

ax.text(0.5, 0.65, example1, ha='center', fontsize=10, family='monospace',
       transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='#ffcccc', alpha=0.7))
ax.text(0.5, 0.48, example2, ha='center', fontsize=10, family='monospace',
       transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='#ccffcc', alpha=0.7))

ax.text(0.5, 0.3, 'Same current word "bank"\nBut different context needed\nNot Markov!',
       ha='center', fontsize=10, style='italic', transform=ax.transAxes)

ax.text(0.5, 0.08, 'Solution: Make state = (context, current_word)\nNow it\'s Markov!',
       ha='center', fontsize=10, fontweight='bold', transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

plt.tight_layout()
plt.show()

<a name="markov-chains"></a>
## Markov Chains

A **Markov chain** is a sequence of states where transitions follow the Markov property.

### Definition

A finite Markov chain consists of:
- **State space**: S = {1, 2, ..., n}
- **Transition matrix**: P where P_ij = P(s_{t+1} = j | s_t = i)

**Properties**:
- Each row sums to 1 (probabilities from a state sum to 1)
- Entries are non-negative
- Memoryless: next state depends only on current state

### Example: Weather

Three weather states: Sunny (0), Cloudy (1), Rainy (2)

Transition matrix P:
```
        Sunny  Cloudy  Rainy
Sunny  [0.7    0.2     0.1  ]  (sunny → 70% sunny, 20% cloudy, 10% rainy)
Cloudy [0.3    0.4     0.3  ]  (cloudy → 30% sunny, 40% cloudy, 30% rainy)
Rainy  [0.2    0.3     0.5  ]  (rainy → 20% sunny, 30% cloudy, 50% rainy)
```

In [ ]:
# Weather Markov Chain example
P_weather = np.array([
    [0.7, 0.2, 0.1],  # Sunny
    [0.3, 0.4, 0.3],  # Cloudy
    [0.2, 0.3, 0.5]   # Rainy
])

states_weather = ['Sunny', 'Cloudy', 'Rainy']

print("Weather Transition Matrix:")
df = pd.DataFrame(P_weather, index=states_weather, columns=states_weather)
print(df)
print(f"\nRow sums (should be 1.0): {P_weather.sum(axis=1)}")

# Simulate weather sequences
print("\n" + "="*60)
print("Simulating weather sequences")
print("="*60)

def simulate_markov_chain(P, start_state, n_steps):
    """Simulate a Markov chain."""
    trajectory = [start_state]
    current = start_state
    
    for _ in range(n_steps):
        # Sample next state according to P[current]
        next_state = np.random.choice(len(P), p=P[current])
        trajectory.append(next_state)
        current = next_state
    
    return trajectory

# Run simulations
np.random.seed(42)
trajectories = []
for i in range(5):
    traj = simulate_markov_chain(P_weather, start_state=0, n_steps=10)
    weather_names = [states_weather[s] for s in traj]
    trajectories.append(weather_names)
    print(f"Sequence {i+1}: {' → '.join(weather_names)}")

print("\nNotice: Each next state is determined only by current state!")

<a name="mdp-definition"></a>
## Markov Decision Processes

An **MDP** extends a Markov chain by adding **actions** and **rewards**. Now the agent can influence state transitions.

### Formal Definition

A finite MDP is a tuple **(S, A, P, R, γ)** where:

- **S**: Finite set of states
- **A**: Finite set of actions
- **P(s', r | s, a)**: Probability of transition to state s' with reward r, given current state s and action a
- **R(s, a)** or **R(s, a, s')**: Expected reward function
- **γ** ∈ [0, 1]: Discount factor

### Key Insight

Unlike Markov chains (passive observation), in an MDP:
- The agent **chooses** actions
- Actions **influence** state transitions
- Rewards **motivate** good actions
- The goal is to find the **optimal policy** π* that maximizes cumulative reward

In [ ]:
# Define an MDP class
class MDP:
    """Formal MDP representation."""
    
    def __init__(self, name, states, actions, P, R, gamma=0.9):
        """
        Args:
            name: MDP name
            states: List of state names
            actions: List of action names
            P: Dict mapping (s, a) → Dict[s' → probability]
            R: Dict mapping (s, a, s') → reward
            gamma: Discount factor
        """
        self.name = name
        self.S = states
        self.A = actions
        self.P = P  # Transition probabilities
        self.R = R  # Reward function
        self.gamma = gamma
        
        # Create state and action indices
        self.state_to_idx = {s: i for i, s in enumerate(states)}
        self.action_to_idx = {a: i for i, a in enumerate(actions)}
        self.idx_to_state = {i: s for i, s in enumerate(states)}
        self.idx_to_action = {i: a for i, a in enumerate(actions)}
    
    def get_transition_prob(self, state, action, next_state):
        """Get P(s' | s, a)."""
        if (state, action) in self.P:
            return self.P[(state, action)].get(next_state, 0.0)
        return 0.0
    
    def get_reward(self, state, action, next_state):
        """Get R(s, a, s')."""
        if (state, action, next_state) in self.R:
            return self.R[(state, action, next_state)]
        return 0.0
    
    def get_expected_reward(self, state, action):
        """Get expected reward E[R(s, a)]."""
        total_reward = 0.0
        for next_state in self.S:
            prob = self.get_transition_prob(state, action, next_state)
            reward = self.get_reward(state, action, next_state)
            total_reward += prob * reward
        return total_reward


print("✅ MDP class defined")

<a name="examples"></a>
## Classic MDP Examples

### Example 1: Student MDP

A student deciding how much to study before an exam.

**States**:
- S: Short study session completed
- L: Long study session completed

**Actions**:
- Study: Study hard
- Relax: Relax and don't study

**Transitions & Rewards**:
- Study in S: 90% stay in S (reward -1), 10% go to L (reward -1)
- Study in L: Stay in L with reward 10 (pass exam!)
- Relax in S: 100% return to S with reward +1 (enjoy rest)
- Relax in L: 100% return to S with reward +1 (vacation, but forget!)

In [ ]:
# Create Student MDP
student_mdp = MDP(
    name="Student",
    states=['S', 'L'],
    actions=['Study', 'Relax'],
    P={
        ('S', 'Study'): {'S': 0.9, 'L': 0.1},
        ('S', 'Relax'): {'S': 1.0},
        ('L', 'Study'): {'L': 1.0},
        ('L', 'Relax'): {'S': 1.0}
    },
    R={
        ('S', 'Study', 'S'): -1,
        ('S', 'Study', 'L'): -1,
        ('S', 'Relax', 'S'): 1,
        ('L', 'Study', 'L'): 10,
        ('L', 'Relax', 'S'): 1
    },
    gamma=0.9
)

print("Student MDP:")
print(f"States: {student_mdp.S}")
print(f"Actions: {student_mdp.A}")
print(f"Discount factor: {student_mdp.gamma}")

# Display transition probabilities
print("\nTransition Probabilities:")
for state in student_mdp.S:
    for action in student_mdp.A:
        probs = student_mdp.P.get((state, action), {})
        print(f"  P(· | S={state}, a={action}): {probs}")

print("\nExpected Rewards:")
for state in student_mdp.S:
    for action in student_mdp.A:
        exp_reward = student_mdp.get_expected_reward(state, action)
        print(f"  E[R(S={state}, a={action})]: {exp_reward:.2f}")

<a name="bellman-equations"></a>
## Bellman Equations

The **Bellman equations** are recursive relationships that form the foundation of all RL algorithms. They decompose the value of a state into immediate reward plus future value.

### State Value Function (Bellman Expectation)

$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s', r} P(s', r | s, a) [r + \gamma V^\pi(s')]$$

**Interpretation**: The value of a state is the expected immediate reward plus γ times the value of the next state.

### Action Value Function (Q-function)

$$Q^\pi(s, a) = \sum_{s', r} P(s', r | s, a) [r + \gamma V^\pi(s')]$$

**Interpretation**: The value of taking action a in state s is the expected immediate reward plus γ times the value of the next state.

### Optimal Value Functions

The **optimal** value functions satisfy:

$$V^*(s) = \max_a \sum_{s', r} P(s', r | s, a) [r + \gamma V^*(s')]$$

$$Q^*(s, a) = \sum_{s', r} P(s', r | s, a) [r + \gamma \max_{a'} Q^*(s', a')]$$

### Why They're Powerful

Bellman equations connect values of related states. This **structure** enables:
- **Dynamic programming**: Solve one state using values of other states
- **Convergence guarantees**: Iterative algorithms provably converge
- **Efficient computation**: Avoid simulating all trajectories

In [ ]:
# Visualize Bellman backup for Student MDP
fig, ax = plt.subplots(figsize=(12, 6))

# Title
ax.text(0.5, 0.95, 'Bellman Backup Diagram (Student MDP)', ha='center', fontsize=14, fontweight='bold',
        transform=ax.transAxes)

# State S
ax.add_patch(Circle((0.2, 0.7), 0.08, transform=ax.transAxes, color='lightblue', 
                    ec='black', linewidth=2))
ax.text(0.2, 0.7, 'S', ha='center', va='center', transform=ax.transAxes,
       fontsize=12, fontweight='bold')
ax.text(0.2, 0.55, 'Current state', ha='center', transform=ax.transAxes, fontsize=10)

# Action: Study
ax.add_patch(FancyBboxPatch((0.05, 0.38), 0.12, 0.05, transform=ax.transAxes,
                           boxstyle='round,pad=0.01', facecolor='lightyellow', ec='black'))
ax.text(0.11, 0.405, 'Study', ha='center', va='center', transform=ax.transAxes, fontsize=9)

# Action: Relax
ax.add_patch(FancyBboxPatch((0.23, 0.38), 0.12, 0.05, transform=ax.transAxes,
                           boxstyle='round,pad=0.01', facecolor='lightyellow', ec='black'))
ax.text(0.29, 0.405, 'Relax', ha='center', va='center', transform=ax.transAxes, fontsize=9)

# Next states
ax.add_patch(Circle((0.03, 0.15), 0.06, transform=ax.transAxes, color='lightcoral',
                   ec='black', linewidth=2))
ax.text(0.03, 0.15, 'S\'', ha='center', va='center', transform=ax.transAxes,
       fontsize=10, fontweight='bold')
ax.text(0.03, 0.03, "V(S')=0.5", ha='center', transform=ax.transAxes, fontsize=9, family='monospace')

ax.add_patch(Circle((0.2, 0.15), 0.06, transform=ax.transAxes, color='lightcoral',
                   ec='black', linewidth=2))
ax.text(0.2, 0.15, 'L\'', ha='center', va='center', transform=ax.transAxes,
       fontsize=10, fontweight='bold')
ax.text(0.2, 0.03, "V(L')=5.0", ha='center', transform=ax.transAxes, fontsize=9, family='monospace')

ax.add_patch(Circle((0.37, 0.15), 0.06, transform=ax.transAxes, color='lightcoral',
                   ec='black', linewidth=2))
ax.text(0.37, 0.15, 'S\'', ha='center', va='center', transform=ax.transAxes,
       fontsize=10, fontweight='bold')
ax.text(0.37, 0.03, "V(S')=0.5", ha='center', transform=ax.transAxes, fontsize=9, family='monospace')

# Arrows and labels
ax.annotate('', xy=(0.08, 0.38), xytext=(0.17, 0.7),
           arrowprops=dict(arrowstyle='->', lw=2), xycoords=ax.transAxes)
ax.text(0.12, 0.54, 'p=0.9\nr=-1', ha='center', fontsize=9, family='monospace')

ax.annotate('', xy=(0.11, 0.21), xytext=(0.08, 0.38),
           arrowprops=dict(arrowstyle='->', lw=2), xycoords=ax.transAxes)
ax.text(0.05, 0.28, '0.9', ha='center', fontsize=8, family='monospace')

ax.annotate('', xy=(0.2, 0.21), xytext=(0.08, 0.38),
           arrowprops=dict(arrowstyle='->', lw=2), xycoords=ax.transAxes)
ax.text(0.14, 0.28, '0.1', ha='center', fontsize=8, family='monospace')

ax.annotate('', xy=(0.37, 0.21), xytext=(0.29, 0.38),
           arrowprops=dict(arrowstyle='->', lw=2), xycoords=ax.transAxes)
ax.text(0.33, 0.28, 'p=1.0\nr=1', ha='center', fontsize=9, family='monospace')

# Bellman equation
ax.text(0.5, 0.5, 'V(S) = E[r + γV(S\')]', ha='center', fontsize=11, family='monospace',
       transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

ax.text(0.5, 0.35, 'For Study: V(S) = (-1) + 0.9×0.5 + 0.1×5.0\n           = -1 + 0.45 + 0.5 = -0.05',
       ha='center', fontsize=10, family='monospace', transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='#ffffcc', alpha=0.8))

ax.text(0.5, 0.2, 'For Relax: V(S) = 1 + 0.9×0.5\n           = 1 + 0.45 = 1.45',
       ha='center', fontsize=10, family='monospace', transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='#ccffcc', alpha=0.8))

ax.text(0.5, 0.08, '✓ Relax is better! V(S) = max(-0.05, 1.45) = 1.45',
       ha='center', fontsize=10, fontweight='bold', transform=ax.transAxes)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

plt.tight_layout()
plt.show()

<a name="mdp-solver"></a>
## From-Scratch MDP Solver

Given an MDP (S, A, P, R, γ), we can compute optimal value functions using the **Value Iteration** algorithm, which repeatedly applies the Bellman equations until convergence.

In [ ]:
class MDPSolver:
    """Solve MDPs using value iteration."""
    
    def __init__(self, mdp, epsilon=1e-6, max_iterations=1000):
        self.mdp = mdp
        self.epsilon = epsilon
        self.max_iterations = max_iterations
        self.V = {s: 0.0 for s in mdp.S}  # Value function
        self.Q = {}  # Q-function
        self.policy = {s: None for s in mdp.S}  # Optimal policy
    
    def value_iteration(self):
        """Compute optimal value function using value iteration."""
        iteration_history = []
        
        for iteration in range(self.max_iterations):
            V_old = self.V.copy()
            
            # Bellman update for each state
            for state in self.mdp.S:
                max_q = -np.inf
                
                # Find action with highest value
                for action in self.mdp.A:
                    # Q(s, a) = E[r + γ V(s')]
                    q_value = 0.0
                    for next_state in self.mdp.S:
                        prob = self.mdp.get_transition_prob(state, action, next_state)
                        reward = self.mdp.get_reward(state, action, next_state)
                        q_value += prob * (reward + self.mdp.gamma * V_old[next_state])
                    
                    if q_value > max_q:
                        max_q = q_value
                        self.policy[state] = action
                
                self.V[state] = max_q
            
            # Check convergence
            delta = max(abs(self.V[s] - V_old[s]) for s in self.mdp.S)
            iteration_history.append(delta)
            
            if delta < self.epsilon:
                print(f"✅ Converged in {iteration + 1} iterations (delta={delta:.2e})")
                return iteration_history
        
        print(f"⚠️  Did not converge after {self.max_iterations} iterations (final delta={delta:.2e})")
        return iteration_history
    
    def get_optimal_policy(self):
        """Return the optimal policy."""
        return self.policy.copy()
    
    def get_value_function(self):
        """Return the optimal value function."""
        return self.V.copy()


# Solve Student MDP
print("Solving Student MDP using Value Iteration...\n")
solver = MDPSolver(student_mdp, epsilon=1e-6)
history = solver.value_iteration()

print(f"\nOptimal Value Function:")
for state, value in solver.get_value_function().items():
    print(f"  V({state}) = {value:.4f}")

print(f"\nOptimal Policy:")
for state, action in solver.get_optimal_policy().items():
    print(f"  π({state}) = {action}")

## Conclusion

### What We Learned

1. **Markov property**: Future is independent of past given present
2. **Markov chains**: State sequences without agents
3. **MDPs**: Formal model for sequential decision-making
4. **Bellman equations**: Recursive value decomposition
5. **Value iteration**: Algorithm to solve MDPs optimally

### Why It Matters

- MDPs formalize problem definition
- Bellman equations enable efficient algorithms
- Value iteration finds optimal solutions
- Every RL algorithm solves an MDP (or a variation)

### Next: Lesson 1B

In the practical lesson, we'll implement:
- **Policy evaluation**: Compute V(s) for a given policy
- **Policy iteration**: Find optimal policy iteratively
- **Value iteration** (detailed implementation)
- Use **FrozenLake** from Gymnasium

The mathematical foundation is now solid. Time to build algorithms! 🚀